# 07 Experiment Design

## Goal

Design future A/B tests using the product analytics findings without fabricating experiment results.

## Setup

Connect to the local DuckDB database created by the transformation pipeline.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import duckdb
import pandas as pd
import matplotlib.pyplot as plt

from product_growth_analytics.notebook_utils import connect, query_df, read_sql

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.4f}".format)
con = connect()

## Concept

This notebook designs experiments. It does not claim experiment results exist. Observational analysis can suggest opportunities; only randomized experiments can support causal claims.

In [2]:
import math
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

## Experiment Candidate 1: Cart Reminder

Hypothesis: sending a timely cart reminder to eligible cart users will increase cart-to-purchase conversion without lowering average order value.

In [3]:
baseline_conversion = 0.10
minimum_detectable_lift = 0.05
alpha = 0.05
power = 0.80

control_rate = baseline_conversion
treatment_rate = baseline_conversion * (1 + minimum_detectable_lift)
effect_size = proportion_effectsize(control_rate, treatment_rate)
analysis = NormalIndPower()
sample_size_per_group = math.ceil(analysis.solve_power(
    effect_size=effect_size,
    alpha=alpha,
    power=power,
    ratio=1,
    alternative="two-sided",
))

sample_size_per_group

57756

## Experiment Design Template

| Field | Definition |
| --- | --- |
| Business problem | High-intent cart users may not complete purchase. |
| Hypothesis | A cart reminder increases cart-to-purchase conversion. |
| Primary metric | Cart-to-purchase rate among eligible cart users. |
| Secondary metrics | Purchase revenue per cart user, repeat purchase rate. |
| Guardrails | AOV, unsubscribe/notification opt-out if available, refund rate if available. |
| Randomization unit | User ID. |
| Target population | Users with cart event and no purchase within defined window. |
| Success criteria | Statistically and practically meaningful lift in primary metric with no guardrail harm. |

## Experiment Candidate 2: Category Recommendation

Hypothesis: recommending adjacent categories after first purchase increases repeat purchase rate and revenue per buyer.

## Experiment Candidate 3: Product Detail Page Improvement

Hypothesis: improving product detail information increases view-to-cart rate in low-conversion categories.

## Takeaways

- This notebook does **not** fabricate experiment results. It uses the observational analysis to propose future experiments and define how Product should test them.
- The strongest experiment candidate is a **product discovery / product detail page improvement** because Notebook 03 showed the largest funnel drop-off before cart: view-to-cart is **19.83%**, while cart-to-purchase is much stronger at **66.17%**.
- A cart reminder is still a valid lifecycle experiment, but it targets users who already show intent. It should be measured with cart-to-purchase rate as the primary metric and AOV/revenue per cart user as guardrails.
- A category recommendation experiment is well aligned with Notebook 06 because repeat purchasers engage with broader category and brand breadth. The primary metric should be repeat purchase rate, with revenue per buyer and AOV as secondary/guardrail metrics.
- The illustrative sample-size calculation in this notebook assumes a **10% baseline conversion rate** and a **5% relative lift**, requiring about **57,756 users per group**. Using observed baselines, a 5% relative lift would require roughly **25,856 users per group** for view-to-cart, **3,128 per group** for cart-to-purchase, and **14,862 per group** for repeat purchase.
- Randomization should happen at the **user_id** level so the same user does not see multiple variants across sessions.
- Success should require both statistical significance and business relevance. For example, a lift in conversion is not enough if AOV, revenue per user, or repeat purchase quality declines.

Overall, the recommended experiment path is: first test improvements that move more viewers into cart, then test lifecycle nudges that increase repeat purchase among high-intent or first-time buyers.